In [3]:
import yaml

with open("config.yml") as f:
    config = yaml.safe_load(f)

for k, v in config.items():
    print(f"{k}: {v}")

experiment_name: LSTM_CAMELS_DE_1h_benchmark
path_data: /pfs/work9/workspace/scratch/ka_yh2352-camels_de_1h_data/CAMELS-DE-1h/CAMELS-DE-1h/
path_entities: /pfs/data6/home/ka/ka_iwu/ka_yh2352/Hy2DL-Testing/basins_list.txt
dynamic_input: {'1h': ['discharge_spec_obs', 'air_temperature_mean', 'global_radiation_mean', 'air_pressure_surface_mean', 'relative_humidity_mean', 'wind_speed_mean', 'cloud_cover_mean'], '1D': ['discharge_spec_obs', 'air_temperature_mean', 'global_radiation_mean', 'air_pressure_surface_mean', 'relative_humidity_mean', 'wind_speed_mean', 'cloud_cover_mean']}
target: ['precipitation_mean', 'precipitation_max', 'precipitation_stdev']
static_input: ['area', 'elev_mean', 'p_mean', 'frac_snow']
training_period: ['2004-01-01 01:00:00', '2019-12-31 23:00:00']
validation_period: ['2001-01-01 01:00:00', '2003-12-31 23:00:00']
testing_period: ['2020-01-01 01:00:00', '2024-12-31 23:00:00']
hidden_size: 128
batch_size_training: 256
batch_size_evaluation: 1024
epochs: 20
dropout_r

In [9]:
from pathlib import Path
from hy2dl.utils.config import Config
from hy2dl.datasetzoo.hourlycamelsde import Hourly_CAMELS_DE

cfg_path = Path("config.yml").resolve()
print(cfg_path)  # sanity check — should show .../LSTM_CAMELS_DE_1h_benchmark_seed_100/config.yml, once

cfg = Config(yml_path_or_dict=cfg_path, base_dir=cfg_path.parent)

ds = Hourly_CAMELS_DE(cfg=cfg, time_period="testing", gauge_id="DE110000")
df = ds._read_data(gauge_id="DE110000")
print([c for c in df.columns if "discharge_lead" in c])

/pfs/data6/home/ka/ka_iwu/ka_yh2352/Hy2DL-Testing/results/LSTM_CAMELS_DE_1h_benchmark_seed_100/config.yml
['discharge_lead_12h', 'discharge_lead_24h', 'discharge_lead_36h', 'discharge_lead_48h']


In [6]:
from hy2dl.utils.config import Config
import inspect
print(inspect.signature(Config.__init__))

(self, yml_path_or_dict: dict, base_dir: pathlib.Path, dev_mode: bool = False)


In [10]:
import inspect
from hy2dl.datasetzoo.camelsde import CAMELS_DE
print(inspect.getsource(CAMELS_DE))

class CAMELS_DE(BaseDataset):
    """
    Class to process data from [version 1.0.0 of the CAMELS Germany dataset]
    (https://doi.org/10.5281/zenodo.13837553) by [1]_ [2]_ .

    The class inherits from BaseDataset to execute the operations on how to load and process the data. However here we
    code the _read_attributes and _read_data methods, that specify how we should read the information from CAMELS DE.

    Parameters
    ----------
    cfg : Config
        Configuration file.
    period : {'training', 'validation', 'testing'}
        Defines the period for which the data will be loaded.
    gauge_id : Optional[str | list[str]], default=None
        Id of gauge(s) to be loaded.

    References
    ----------
    .. [1] Loritz, R., Dolich, A., Acuña Espinoza, E., Ebeling, P., Guse, B., Götte, J., Hassler, S. K., Hauffe,
        C., Heidbüchel, I., Kiesel, J., Mälicke, M., Müller-Thomy, H., Stölzle, M., & Tarasova, L. (2024).
        CAMELS-DE: Hydro-meteorological time series an

In [11]:
ds_full = Hourly_CAMELS_DE(cfg=cfg, time_period="testing", gauge_id="DE110000")
# look for an attribute holding the final processed dataframe or tensor, e.g.:
print(ds_full.df.columns if hasattr(ds_full, "df") else "no .df attribute — check ds_full.__dict__")
print(ds_full.__dict__.keys())  # to see what attributes actually exist

no .df attribute — check ds_full.__dict__
dict_keys(['cfg', 'period', 'time_period', 'gauge_id', 'dynamic_input', 'input_per_freq', 'hindcast_input', 'pseudo_forecast_input', 'pseudo_forecast_ar_input', 'forecast_input', 'data_freq', 'data_step', 'start_date', 'end_date', 'warmup_start_date', 'variables_of_interest', 'dataset_in_ram', 'fc_in_ram', 'basin_std'])


In [12]:
print("dynamic_input:", ds_full.dynamic_input)
print("\nvariables_of_interest:", ds_full.variables_of_interest)

dynamic_input: ['discharge_spec_obs', 'air_temperature_mean', 'global_radiation_mean', 'air_pressure_surface_mean', 'relative_humidity_mean', 'wind_speed_mean', 'cloud_cover_mean']

variables_of_interest: ['discharge_spec_obs', 'air_temperature_mean', 'global_radiation_mean', 'air_pressure_surface_mean', 'relative_humidity_mean', 'wind_speed_mean', 'cloud_cover_mean', 'precipitation_mean', 'precipitation_max', 'precipitation_stdev']


In [13]:
print("input_per_freq:", ds_full.input_per_freq)

input_per_freq: {'1h': ['discharge_spec_obs', 'air_temperature_mean', 'global_radiation_mean', 'air_pressure_surface_mean', 'relative_humidity_mean', 'wind_speed_mean', 'cloud_cover_mean'], '1D': ['discharge_spec_obs', 'air_temperature_mean', 'global_radiation_mean', 'air_pressure_surface_mean', 'relative_humidity_mean', 'wind_speed_mean', 'cloud_cover_mean']}


In [14]:
import inspect
from hy2dl.datasetzoo.basedataset import BaseDataset  # adjust path if it lives elsewhere
src = inspect.getsource(BaseDataset)
# search within src for "custom_seq_processing" to see exactly how n_steps/freq_factor are used
print([line for line in src.splitlines() if "custom_seq_processing" in line or "freq_factor" in line or "n_steps" in line])

['        if self.cfg.custom_seq_processing is not None and isinstance(self.cfg.dynamic_input, dict):', '                k: BaseDataset.unique_values(self.cfg.dynamic_input[k]) for k in self.cfg.custom_seq_processing', '            freq_factor: Optional[int] = None,', '            freq_factor: Optional[int], default=None', '            if freq_factor is not None:', '                x_tensor = x_tensor.reshape(b, t // freq_factor, freq_factor, f).mean(dim=2)', '        if self.cfg.custom_seq_processing is None:  # single frequency', '            for subset_name, subset_info in self.cfg.custom_seq_processing.items():', '                subset_length = subset_info["n_steps"] * subset_info["freq_factor"]', '                    freq_factor=subset_info["freq_factor"],', '        elif isinstance(self.cfg.dynamic_input, dict) and self.cfg.custom_seq_processing is None:', '        elif isinstance(self.cfg.dynamic_input, dict) and isinstance(self.cfg.custom_seq_processing, dict):', '            

In [16]:
print(df["discharge_lead_48h"].isna().sum())
print(df["discharge_lead_48h"].tail(100))

48
date
2024-12-27 20:00:00    0.05166
2024-12-27 21:00:00    0.05142
2024-12-27 22:00:00    0.05095
2024-12-27 23:00:00    0.05095
2024-12-28 00:00:00    0.05119
                        ...   
2024-12-31 19:00:00        NaN
2024-12-31 20:00:00        NaN
2024-12-31 21:00:00        NaN
2024-12-31 22:00:00        NaN
2024-12-31 23:00:00        NaN
Name: discharge_lead_48h, Length: 100, dtype: float64


In [17]:
import inspect
from hy2dl.utils.config import Config
src = inspect.getsource(Config)
print([line for line in src.splitlines() if "random_seed" in line.lower() or "seed" in line.lower()])

['        self.logger = get_logger(self.path_save_folder, f"{self.experiment_name}_{self.random_seed}")', '        suffix = f"{self.experiment_name}_seed_{self.random_seed}"', '    def random_seed(self) -> int:', '        if self._cfg.get("random_seed") is None:', '            self._cfg["random_seed"] = int(np.random.uniform(0, 1e6))', '        return self._cfg.get("random_seed")', '    @random_seed.setter', '    def random_seed(self, value: int):', '        self._cfg["random_seed"] = value']
